In [1]:
import pandas as pd

In [2]:
llm_df = pd.read_csv(
    "../data/processed/validation/llm_raw_responses.csv",
    names=["candidate_id", "relevance_score"],
    dtype=str,
    on_bad_lines="skip",
)

master_df = pd.read_csv("../data/processed/validation/master_eval_dataset.csv")


PROCESSED_BATCHES = 50

In [ ]:
llm_df = llm_df[llm_df["candidate_id"] != "candidate_id"].dropna()

llm_df["candidate_id"] = llm_df["candidate_id"].str.strip().astype(int)
llm_df["relevance_score"] = llm_df["relevance_score"].str.strip().astype(int)

master_df = master_df.sort_values("anchor_id").reset_index(drop=True)

processed_anchors = master_df["anchor_id"].unique()[:PROCESSED_BATCHES]

master_batches = master_df[master_df["anchor_id"].isin(processed_anchors)].copy()


expected_rows = len(master_batches)
actual_rows = len(llm_df)

print(f"Ожидаем кандидатов в {PROCESSED_BATCHES} батчах: {expected_rows}")
print(f"Получено ответов от LLM: {actual_rows}")

if expected_rows == actual_rows:
    print("Количество строк совпадает! Нейронка ничего не забыла.")

    # Проверяем порядок внутри батчей
    mismatches = (
        master_batches["candidate_id"].values != llm_df["candidate_id"].values
    ).sum()

    if mismatches > 0:
        raise "порядок вакансий был перепутан!!!"

    print("Порядок ID идеальный! Галлюцинаций нет.")
    master_batches["relevance_score"] = llm_df["relevance_score"].values
    print("🎉 Датасет успешно восстановлен!")

else:
    print("ОШИБКА: Нейронка потеряла или выдумала лишние строки!")
    print(f"Разница в {abs(expected_rows - actual_rows)} строк(и).")


Ожидаем кандидатов в 50 батчах: 2263
Получено ответов от LLM: 2263
Количество строк совпадает! Нейронка ничего не забыла.
Порядок ID идеальный! Галлюцинаций нет.
🎉 Датасет успешно восстановлен!


In [6]:
master_batches.to_csv(
    "../data/processed/validation/ground_truth.csv",
    index=False,
    encoding="utf-8",
)

In [5]:
master_batches.head(5)

,anchor_id,anchor_title,anchor_text,candidate_id,candidate_title,candidate_text,source,relevance_score
0,43200117,"Старший разработчик C#, Поиск / Поисковые нави...","Должность: Старший разработчик C#, Поиск / Пои...",48299803,"Разработчик C#, Собственные продажи","Должность: Разработчик C#, Собственные продажи...",E5,2
1,43200117,"Старший разработчик C#, Поиск / Поисковые нави...","Должность: Старший разработчик C#, Поиск / Пои...",49419678,"Разработчик JavaScript, Инструменты поисковой ...","Должность: Разработчик JavaScript, Инструменты...",TF-IDF,0
2,43200117,"Старший разработчик C#, Поиск / Поисковые нави...","Должность: Старший разработчик C#, Поиск / Пои...",49502241,"Разработчик C#, Логистика/Биллинг","Должность: Разработчик C#, Логистика/Биллинг\n...",E5,2
3,43200117,"Старший разработчик C#, Поиск / Поисковые нави...","Должность: Старший разработчик C#, Поиск / Пои...",49176098,"Перформанс инженер, Поиск /Поисковый рантайм","Должность: Перформанс инженер, Поиск /Поисковы...",TF-IDF,1
4,43200117,"Старший разработчик C#, Поиск / Поисковые нави...","Должность: Старший разработчик C#, Поиск / Пои...",49437730,С++ разработчик (отдел поисковых технологий),Должность: С++ разработчик (отдел поисковых те...,TF-IDF,1
